In [0]:
df_adm = spark.table("capstone.silver.admissions")
df_pat = spark.table("capstone.silver.patients")
df_doc = spark.table("capstone.silver.doctors")
df_hosp = spark.table("capstone.silver.hospitals")
df_lab = spark.table("capstone.silver.lab_results")
df_bill = spark.table("capstone.silver.billing")

In [0]:
# =========================================
# FILTER VALID DATA
# =========================================
from pyspark.sql.functions import col
df_adm = df_adm.filter(
    col("discharge_date") >= col("admission_date")
)

df_pat = df_pat.filter(
    col("age") > 0
)

df_bill = df_bill.filter(
    col("total_charges") >= col("insurance_covered_amount")
)

Creating Fact Table

In [0]:
df_fact_adm = df_adm.alias("adm") \
    .join(df_pat.alias("pat"), col("adm.patient_id") == col("pat.patient_id"), "left") \
    .join(df_doc.alias("doc"), col("adm.attending_doctor_id") == col("doc.doctor_id"), "left") \
    .join(df_hosp.alias("hosp"), col("adm.hospital_id") == col("hosp.hospital_id"), "left")

In [0]:
df_fact_adm = df_fact_adm.select(
    col("adm.admission_id"),
    col("adm.patient_id"),
    col("adm.hospital_id"),
    col("adm.attending_doctor_id"),
    col("adm.department"),
    col("adm.admission_date"),
    col("adm.discharge_date"),
    col("adm.admission_type"),
    col("adm.LOS"),
    col("adm.risk_flag"),
    col("adm.readmission_flag")   # 🔥 NEW COLUMN
)

In [0]:
df_adm.printSchema()

root
 |-- admission_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- diagnosis_code: integer (nullable = true)
 |-- attending_doctor_id: integer (nullable = true)
 |-- LOS: integer (nullable = true)
 |-- risk_flag: integer (nullable = true)
 |-- readmission_flag: integer (nullable = true)



In [0]:
df_fact_adm.display()

admission_id,patient_id,hospital_id,attending_doctor_id,department,admission_date,discharge_date,admission_type,LOS,risk_flag,readmission_flag


In [0]:
from pyspark.sql.functions import col

df_fact_adm.filter(col("admission_id").isNull()).count()

0

In [0]:
df_fact_adm = df_fact_adm.dropDuplicates(["admission_id"])

admission Duration Category

In [0]:
from pyspark.sql.functions import when

df_fact_adm = df_fact_adm.withColumn(
    "stay_category",
    when(col("LOS") <= 3, "Short")
    .when((col("LOS") > 3) & (col("LOS") <= 10), "Medium")
    .otherwise("Long")
)

High Risk Flag

In [0]:
df_fact_adm = df_fact_adm.withColumn(
    "high_risk",
    (col("LOS") > 7).cast("int")
)

In [0]:
df_fact_adm.write.format("delta").option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("capstone.gold.fact_admissions")

FACT Billing

In [0]:
df_fact_billing = df_bill.select(
    "bill_id",
    "admission_id",
    "total_charges",
    "insurance_covered_amount",
    "patient_payable",
    "claim_status"
)

In [0]:
df_fact_billing.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone.gold.fact_billing")

In [0]:
df_dim_patient = df_pat.dropDuplicates(["patient_id"]).select(
    "patient_id",
    "age",
    "gender",
    "city",
    "insurance_type",
    "chronic_condition_flag"
)

In [0]:
df_dim_patient.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone.gold.dim_patient")

In [0]:
df_dim_doctor = df_doc.dropDuplicates(["doctor_id"]).select(
    "doctor_id",
    "department",
    "specialization",
    "experience_years"
)


In [0]:
df_dim_doctor.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone.gold.dim_doctor")

In [0]:
df_dim_hospital = df_hosp.dropDuplicates(["hospital_id"]).select(
    "hospital_id",
    "hospital_name",
    "city",
    "capacity_beds"
)

In [0]:
df_dim_hospital.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone.gold.dim_hospital")

In [0]:
spark.table("capstone.gold.fact_admissions").display()
spark.table("capstone.gold.fact_billing").display()

admission_id,patient_id,hospital_id,attending_doctor_id,department,admission_date,discharge_date,LOS,risk_flag,admission_type,stay_category,high_risk,readmission_flag


bill_id,admission_id,total_charges,insurance_covered_amount,patient_payable,claim_status
600000,880234,162776.95,95591.94,67185.01000000001,Rejected
600001,1194985,191927.02,166612.77,25314.25,Rejected
600002,808128,45840.12,21639.07,24201.050000000003,Approved
600003,1179355,187850.34,106479.65,81370.69,Rejected
600004,864102,64128.56,34810.06,29318.5,Approved
600005,788253,51748.27,20638.05,31110.219999999998,Pending
600006,1035719,90233.68,32612.29,57621.38999999999,Rejected
600007,866475,188993.7,80887.79,108105.91000000002,Pending
600008,1108238,190358.56,136174.73,54183.82999999999,Approved
600009,939262,23727.96,14344.03,9383.929999999998,Approved


In [0]:
from pyspark.sql import Row
from datetime import datetime

record_count =  df_fact_adm.count()

log_data = [Row(
    pipeline_name="healthcare_pipeline",
    layer="gold",
    run_time=datetime.now(),
    status="SUCCESS",
    records_processed=record_count
)]

df_log = spark.createDataFrame(log_data)

df_log.write.mode("append").saveAsTable("capstone.metadata.pipeline_log")